In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração para mostrar números sem notação científica (ex: 1,000,000.00)
pd.options.display.float_format = '{:,.2f}'.format

# --- FUNÇÕES DE LIMPEZA ---
def limpar_valor(valor):
    """Transforma '124M €' em 124000000.0"""
    if pd.isna(valor): return 0
    if isinstance(valor, (int, float)): return valor
    valor = str(valor).replace('€', '').strip()
    if 'M' in valor: return float(valor.replace('M', '')) * 1_000_000
    if 'K' in valor: return float(valor.replace('K', '')) * 1_000
    return float(valor)

def limpar_idade(valor):
    """Transforma '7 fev 1999 (26)' em 26"""
    if pd.isna(valor): return 0
    valor = str(valor)
    if '(' in valor:
        return int(valor.split('(')[-1].replace(')', ''))
    return 0

print("Bibliotecas e funções carregadas!")

Bibliotecas e funções carregadas!


In [ ]:
# 1. Carregar Jogadores (Verifique o nome do arquivo)
df = pd.read_csv('jogadores/sofascore_bundesliga_geral_final_valor_de_mercado_24_25.csv')
df['Liga'] = 'Bundesliga'

# 2. Aplicar Limpeza
df['Valor_Numerico'] = df['Valor de mercado'].apply(limpar_valor)
df['Idade_Limpa'] = df['Idade'].apply(limpar_idade)
df['Passe_Numerico'] = df['Acerto no passe %'].apply(lambda x: float(str(x).replace('%','')) if pd.notna(x) else 0)

correcao_jogadores = {
    'Florian Wirtz': 'Bayer 04 Leverkusen',
    'Alejandro Grimaldo': 'Bayer 04 Leverkusen',
    'Granit Xhaka': 'Bayer 04 Leverkusen',
    'Jeremie Frimpong': 'Bayer 04 Leverkusen',
    'Victor Okoh Boniface': 'Bayer 04 Leverkusen',
    'Edmond Tapsoba': 'Bayer 04 Leverkusen',
    'Piero Hincapié': 'Bayer 04 Leverkusen',
    'Exequiel Palacios': 'Bayer 04 Leverkusen',
    'Jonathan Tah': 'Bayer 04 Leverkusen',
    'Robert Andrich': 'Bayer 04 Leverkusen',
    'Nathan Tella': 'Bayer 04 Leverkusen',
    'Patrik Schick': 'Bayer 04 Leverkusen',
    'Omar Marmoush': 'Eintracht Frankfurt',   # Estava no City/Outros
    'Willian Pacho': 'Eintracht Frankfurt',
    'Hugo Ekitiké': 'Eintracht Frankfurt',
    'Serhou Guirassy': 'VfB Stuttgart',       # Saiu, mas conta como Stuttgart
    'Waldemar Anton': 'VfB Stuttgart',
    'Hiroki Ito': 'VfB Stuttgart',
    'Chris Führich': 'VfB Stuttgart',
    'Maximilian Mittelstädt': 'VfB Stuttgart',
    'Deniz Undav': 'VfB Stuttgart',
    'Matthijs de Ligt': 'FC Bayern München',
    'Noussair Mazraoui': 'FC Bayern München',
    'Xavi Simons': 'RB Leipzig',
    'Dani Olmo': 'RB Leipzig',
    'Niclas Füllkrug': 'Borussia Dortmund',
    'Mats Hummels': 'Borussia Dortmund',
    'Marco Reus': 'Borussia Dortmund',
    'Ian Maatsen': 'Borussia Dortmund'
}


df['Time_Original'] = df['Time']
df['Time'] = df['Nome'].map(correcao_jogadores).fillna(df['Time'])

print(f"Jogadores carregados: {len(df)}")


Jogadores carregados: 299
Verificação Florian Wirtz: ['Bayer 04 Leverkusen']


In [4]:
# 1. Carregar Tabela
df_tabela = pd.read_csv('tabela/tabela_bundesliga_24_25.csv')

# 2. Limpeza de Colunas (Remove espaços extras nos nomes das colunas)
df_tabela.columns = df_tabela.columns.str.strip()

# 3. Renomear para 'Pos' (Garante que o código ache a coluna)
mapa_colunas = {'Posição': 'Pos', '#': 'Pos', 'Rank': 'Pos', 'RK': 'Pos'}
df_tabela.rename(columns=mapa_colunas, inplace=True)

# 4. Tratar Gols (XX:YY) para Gols Pró e Contra
if 'Gols' in df_tabela.columns:
    gols_split = df_tabela['Gols'].str.split(':', expand=True)
    if len(gols_split.columns) == 2:
        df_tabela['Gols_Pro'] = pd.to_numeric(gols_split[0])
        df_tabela['Gols_Contra'] = pd.to_numeric(gols_split[1])

# 5. CORREÇÃO DOS NOMES DOS TIMES (Para bater com os Jogadores)
# Esquerda: Nome na Tabela | Direita: Nome no arquivo de Jogadores (SofaScore)
mapa_times_tabela = {
    'Bayern': 'FC Bayern München',
    'Bayern München': 'FC Bayern München',
    'Bayer Leverkusen': 'Bayer 04 Leverkusen',
    'Leverkusen': 'Bayer 04 Leverkusen',
    'Dortmund': 'Borussia Dortmund',
    'Leipzig': 'RB Leipzig',
    'RB Leipzig': 'RB Leipzig',
    'Stuttgart': 'VfB Stuttgart',
    'VfB Stuttgart': 'VfB Stuttgart',
    'Frankfurt': 'Eintracht Frankfurt',
    'Eintracht Frankfurt': 'Eintracht Frankfurt',
    'Hoffenheim': 'TSG Hoffenheim',
    'TSG 1899 Hoffenheim': 'TSG Hoffenheim',
    'Heidenheim': '1. FC Heidenheim',
    'Werder Bremen': 'SV Werder Bremen',
    'Freiburg': 'SC Freiburg',
    'Augsburg': 'FC Augsburg',
    'Wolfsburg': 'VfL Wolfsburg',
    'Mainz': '1. FSV Mainz 05',
    'Mainz 05': '1. FSV Mainz 05',
    'M\'gladbach': 'Borussia M\'gladbach',
    'Borussia Mönchengladbach': 'Borussia M\'gladbach',
    'Union Berlin': '1. FC Union Berlin',
    'Bochum': 'VfL Bochum 1848',
    'Köln': '1. FC Köln',
    'Darmstadt': 'SV Darmstadt 98',
    'Holstein Kiel': 'Holstein Kiel',
    'St. Pauli': 'FC St. Pauli'
}
df_tabela['Time'] = df_tabela['Time'].str.strip().replace(mapa_times_tabela)

# 6. Cruzamento (Merge) - Trazendo PTS e Pos para os jogadores
# Primeiro calculamos o valor do elenco
elenco_stats = df.groupby('Time').agg({
    'Valor_Numerico': 'sum',
    'Média das notas Sofascore': 'mean'
}).reset_index().rename(columns={'Valor_Numerico': 'Valor_Total_Elenco', 'Média das notas Sofascore': 'Media_Sofa_Time'})

# Merge Tabela + Elenco
df_times_final = pd.merge(df_tabela, elenco_stats, on='Time', how='inner')

# Merge Jogadores + Tabela Completa
df_final = pd.merge(df, df_times_final[['Time', 'Pos', 'PTS', 'Media_Sofa_Time', 'Valor_Total_Elenco']], on='Time', how='inner')

print(f"Merge concluído! Total de jogadores com dados completos: {len(df_final)}")

Merge concluído! Total de jogadores com dados completos: 185


In [ ]:
# 1. CÁLCULO DO ÍNDICE ATLAS (Performance Individual vs Coletiva)
df_final['Indice_Atlas'] = df_final['Média das notas Sofascore'] - df_final['Media_Sofa_Time']

print("=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===")
display(df_final.sort_values('Indice_Atlas', ascending=False)[['Nome', 'Time', 'Pos', 'Média das notas Sofascore', 'Indice_Atlas']].head(10))

# 2. ANÁLISE DE CONTEXTO (Luxo na Lama)
print("\n=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===")

luxo = df_final[
    (df_final['Valor_Numerico'] > 20_000_000) & 
    (df_final['Pos'] > 9)
].sort_values('Valor_Numerico', ascending=False)

if len(luxo) > 0:
    display(luxo[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])
else:
    print("Nenhum jogador encontrado com esses critérios na Bundesliga.")


print("\n=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===")
diamantes = df_final[
    (df_final['Média das notas Sofascore'] > 7.15) & 
    (df_final['Pos'] > 9)
].sort_values('Média das notas Sofascore', ascending=False)

display(diamantes[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])

=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===


,Nome,Time,Pos,Média das notas Sofascore,Indice_Atlas
0,Omar Marmoush,Eintracht Frankfurt,3,8.16,1.10
32,Florian Wirtz,Bayer 04 Leverkusen,2,7.68,0.60
19,Michael Olise,FC Bayern München,1,7.91,0.58
18,Joshua Kimmich,FC Bayern München,1,7.91,0.58
46,Xavi Simons,RB Leipzig,7,7.58,0.55
59,Franck Honorat,Borussia M'gladbach,10,7.43,0.49
60,Moritz Nicolas,Borussia M'gladbach,10,7.37,0.43
20,Harry Kane,FC Bayern München,1,7.73,0.40
89,Nadiem Amiri,1. FSV Mainz 05,6,7.32,0.38
75,Angelo Stiller,VfB Stuttgart,9,7.38,0.36



=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
121,Mohamed Amoura,VfL Wolfsburg,11,35M €,7.17
127,Konstantinos Koulierakis,VfL Wolfsburg,11,24M €,6.90
106,Chrislain Matsima,FC Augsburg,12,21M €,7.14



=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
59,Franck Honorat,Borussia M'gladbach,10,11.2M €,7.43
60,Moritz Nicolas,Borussia M'gladbach,10,4.8M €,7.37
61,Tim Kleindienst,Borussia M'gladbach,10,16.4M €,7.29
104,Finn Dahmen,FC Augsburg,12,12.5M €,7.26
120,Denis Vavro,VfL Wolfsburg,11,6.5M €,7.19
133,Diogo Leite,1. FC Union Berlin,13,11.6M €,7.18
121,Mohamed Amoura,VfL Wolfsburg,11,35M €,7.17


In [ ]:
def ficha_jogador(parte_do_nome):
    """
    Busca um jogador pelo nome (ou parte dele) e mostra todas as colunas.
    """
    dataframe_alvo = df 
    mask = dataframe_alvo['Nome'].str.lower().str.contains(parte_do_nome.lower(), na=False)
    resultado = dataframe_alvo[mask]
    
    if len(resultado) == 0:
        return f"❌ Nenhum jogador encontrado com o nome '{parte_do_nome}'."
    
    return resultado.T

nome_para_buscar = input("Digite o nome (ou parte do nome) do jogador: ")

display(ficha_jogador(nome_para_buscar))

,36
Link Jogador,https://www.sofascore.com/pt/football/player/l...
Time,1. FC Union Berlin
Valor de mercado,11.6M €
Idade,23 de janeiro de 1999 (26)
Pontos fortes,Duelos no chão; Posicionamento
Pontos fracos,Sem pontos fracos notáveis
Posição,Defender
Gols,1
Gols esperados (xG),1.31
Dribles certos,10


In [10]:

grafico_df = df_times_final.copy()

# Tratamento para evitar erro no gráfico (Divisão por zero ou NaN)
grafico_df['PTS'] = pd.to_numeric(grafico_df['PTS'], errors='coerce').fillna(0)
grafico_df['Valor_Total_Elenco'] = pd.to_numeric(grafico_df['Valor_Total_Elenco'], errors='coerce').fillna(0)
pts_ajustado = grafico_df['PTS'].replace(0, 1) # Evita dividir por zero

grafico_df['Custo_Por_Ponto'] = grafico_df['Valor_Total_Elenco'] / pts_ajustado

# Criando o gráfico
fig = px.scatter(grafico_df, 
                 x='Valor_Total_Elenco', 
                 y='PTS', 
                 text='Time', 
                 size='Custo_Por_Ponto', 
                 color='Pos',
                 color_continuous_scale='Bluered_r', # Azul = Topo, Vermelho = Baixo
                 hover_data=['Custo_Por_Ponto'],
                 title='Eficiência Financeira Bundesliga: Valor do Elenco x Pontos',
                 labels={'Valor_Total_Elenco': 'Investimento Total (€)', 'PTS': 'Pontos Conquistados'})

fig.update_traces(textposition='top center')

fig.update_yaxes(range=[0, 100], autorange=False, dtick=10)
fig.show()

In [ ]:
def scouting_bundesliga(posicao=None, orcamento_max=None, atributo_obrigatorio=None):
    res = df_final.copy()
    
    if posicao:
        res = res[res['Posição'].str.contains(posicao, case=False, na=False)]
    
    if orcamento_max:
        valor = float(orcamento_max) * 1_000_000
        res = res[res['Valor_Numerico'] <= valor]
        
    if atributo_obrigatorio:
        res = res[res['Pontos fortes'].str.contains(atributo_obrigatorio, case=False, na=False)]
        
    cols = ['Nome', 'Time', 'Idade_Limpa', 'Valor de mercado', 'Média das notas Sofascore', 'Pontos fortes']
    return res.sort_values('Média das notas Sofascore', ascending=False)[cols].head(10)

print("🔍 Resultado Scouting:")
display(scouting_bundesliga(posicao='Midfielder', orcamento_max=10, atributo_obrigatorio="Armador"))

🔍 Resultado Scouting (Meias, Max 40M, Bom de Passe):


,Nome,Time,Idade_Limpa,Valor de mercado,Média das notas Sofascore,Pontos fortes
123,Maximilian Arnold,VfL Wolfsburg,31,4.7M €,7.13,Armador; Cobranças direta de falta; Posicionam...
151,Pascal Groß,Borussia Dortmund,34,4.5M €,7.13,Armador
63,Kevin Stöger,Borussia M'gladbach,32,3.6M €,7.06,Armador
178,Matúš Bero,VfL Bochum 1848,30,2.9M €,7.03,Alta pressão; Armador
10,Mario Götze,Eintracht Frankfurt,33,4.1M €,6.95,Alta pressão; Armador
134,Christopher Trimmel,1. FC Union Berlin,38,555K €,6.93,Armador
